# Lab 13: YOLO Training

**Module 13** | Read `notes/13-yolo-map.pdf` before running this notebook.

Fine-tune YOLOv8n on COCO128 and read off mAP@0.5 from the validation metrics.

## How to use this notebook

1. **Run cells top to bottom** (Shift+Enter). Do not skip markdown cells; they explain the code.
2. **Read before you run.** Each section tells you what the next code block does, which terms mean, and what the printed output should look like.
3. **Use the comments in code.** Lines starting with `#` explain what each step does in plain language.
4. **Check the output.** After each code cell, compare your printed numbers to the "What to look for" notes.

Code is complete. You are not expected to fill in missing sections or debug skeleton code.


## Runtime device (CPU or GPU)

**What this section does:** PyTorch can run math on your **CPU** (the main processor) or on an **NVIDIA GPU** through **CUDA**. GPUs are faster for neural networks because they run many small matrix operations in parallel.

**Key terms:**
- **CPU:** Always available. Training works but can be slower on large models.
- **GPU / CUDA:** Optional hardware acceleration. If you have an NVIDIA GPU and drivers installed, PyTorch will use it automatically.
- **VRAM:** GPU memory. If you run out, reduce batch size.

The next cell prints which device is active so you know what to expect for training speed.


In [ ]:
import torch

# Pick GPU if CUDA is available; otherwise fall back to CPU (labs still run).
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    props = torch.cuda.get_device_properties(0)
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {props.total_memory / 1e9:.1f} GB")
else:
    print("Using CPU (no CUDA GPU detected).")
    print("All labs still run; training may take longer than on a GPU.")


## YOLOv8 training on COCO128

**YOLO** (You Only Look Once) is a family of object detectors that predict all bounding boxes in a single forward pass through the network. Older pipelines used separate stages (propose regions, then classify each region). YOLO divides the image into a grid and predicts boxes and class probabilities directly, which makes inference fast.

**Fine-tuning pretrained weights** means starting from a model already trained on a large dataset (here `yolov8n.pt` trained on COCO) and continuing training on your smaller dataset (COCO128). The model keeps general visual features and adapts its detection head to the new data.

COCO128 is a tiny 128-image subset of COCO bundled for smoke tests. On first run Ultralytics downloads both weights and images automatically.


## Key terms for this lab

| Term | Meaning |
|------|---------|
| **YOLO** | You Only Look Once. A single-pass detector that predicts bounding boxes and class labels in one network forward pass. |
| **Bounding box** | Rectangle `(x1, y1, x2, y2)` around a detected object. |
| **IoU** | Intersection over Union. Overlap between predicted and ground-truth boxes. Used to decide whether a prediction is a true positive. |
| **mAP** | mean Average Precision. Summarizes detection quality across classes. mAP@0.5 uses IoU threshold 0.5; mAP@0.5:0.95 averages across stricter thresholds and is the main COCO benchmark. |
| **Precision** | Of all predicted boxes, the fraction that are correct (not false alarms). |
| **Recall** | Of all real objects, the fraction the model found (not missed). |
| **Fine-tuning** | Continuing training from pretrained weights on a new dataset instead of training from random initialization. |

Refer back to this table whenever a term appears in code or output.


### Step 1: Training configuration and fine-tune

**What this section does:** Prints hyperparameters, loads `yolov8n.pt` (nano variant, smallest and fastest), and calls `model.train()` on COCO128 for 10 epochs.

**Why we do it:** Starting from pretrained weights is standard practice. The backbone already knows edges, textures, and common shapes; fine-tuning adjusts the detection layers for your data. Ten epochs on 128 images is a smoke test, not a production training run.


**What to look for in the output**

- Configuration block lists model, data, epochs, image size, batch, and device.
- Ultralytics prints download messages on first run (weights and COCO128).
- Training progress bars appear for 10 epochs.
- No crash due to missing CUDA (CPU training is slower but valid).


In [ ]:
from ultralytics import YOLO

TRAIN_CONFIG = {
    "model": "yolov8n.pt",  # Pretrained nano weights (downloads on first use).
    "data": "coco128.yaml",  # Tiny 128-image COCO subset for quick experiments.
    "epochs": 10,
    "imgsz": 640,
    "batch": 16,
    "device": str(device),
}

print("YOLOv8 training configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")
print()

# Load pretrained checkpoint; Ultralytics will fine-tune rather than train from scratch.
model = YOLO(TRAIN_CONFIG["model"])
train_results = model.train(
    data=TRAIN_CONFIG["data"],
    epochs=TRAIN_CONFIG["epochs"],
    imgsz=TRAIN_CONFIG["imgsz"],
    batch=TRAIN_CONFIG["batch"],
    device=TRAIN_CONFIG["device"],
    verbose=True,
)


### Step 2: Training results

**What this section does:** Prints the run directory where weights and plots were saved, plus fitness and metric summaries returned by the trainer.

**Why we do it:** `save_dir` tells you where to find `best.pt` and training curves. Fitness is Ultralytics' combined score used to pick the best checkpoint.


**What to look for in the output**

- `Training complete.` message.
- `Save directory` path under `runs/detect/`.
- Optional `Best fitness` and `Final fitness` values (finite numbers, not None).
- Key metrics from `results_dict` if available.


In [ ]:
print("Training complete.")
print()
print("Run artifacts:")
print(f"  Save directory: {train_results.save_dir}")
if hasattr(train_results, "best_fitness") and train_results.best_fitness is not None:
    print(f"  Best fitness:   {train_results.best_fitness:.4f}")
if hasattr(train_results, "fitness") and train_results.fitness is not None:
    print(f"  Final fitness:  {train_results.fitness:.4f}")

results_dict = train_results.results_dict if hasattr(train_results, "results_dict") else {}
if results_dict:
    print()
    print("Key metrics from training results:")
    for key in sorted(results_dict.keys()):
        val = results_dict[key]
        if isinstance(val, float):
            print(f"  {key}: {val:.4f}")
        else:
            print(f"  {key}: {val}")


### Step 3: Validation metrics (mAP explained)

**What this section does:** Calls `model.val()` to re-run the validation split and print box metrics including mAP@0.5 and mAP@0.5:0.95.

**Why we do it:** mAP is how detection papers compare models. A prediction counts as correct only if its class matches *and* its box overlaps the ground truth enough (IoU above a threshold). mAP@0.5 is lenient (50% overlap is enough). mAP@0.5:0.95 averages across stricter thresholds, so it rewards precise box placement.


**What to look for in the output**

- mAP@0.5 and mAP@0.5:0.95 are finite floats (often modest after only 10 epochs on 128 images).
- Precision and recall between 0 and 1.
- Top 5 per-class mAP values if per-class data is available.
- Values are not NaN.


In [ ]:
metrics = model.val()

print("Validation metrics (detailed):")
print("=" * 50)
print(f"  mAP@0.5:      {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95: {metrics.box.map:.4f}")
print(f"  Precision:    {metrics.box.mp:.4f}")
print(f"  Recall:       {metrics.box.mr:.4f}")
print()

if hasattr(metrics.box, "maps") and metrics.box.maps is not None:
    per_class = metrics.box.maps
    print(f"  Per-class mAP@0.5:0.95 available for {len(per_class)} classes")
    top_indices = sorted(range(len(per_class)), key=lambda i: per_class[i], reverse=True)[:5]
    print("  Top 5 classes by mAP@0.5:0.95:")
    names = metrics.names if hasattr(metrics, "names") else {}
    for idx in top_indices:
        name = names.get(idx, f"class_{idx}")
        print(f"    {name}: {per_class[idx]:.4f}")

print("=" * 50)


### Step 4: Inference on a sample image

**What this section does:** Runs the fine-tuned model on one local JPG and prints each predicted box with class name, confidence, and coordinates. YOLO performs this in a single forward pass.

**Why we do it:** Metrics summarize average quality; inference on a real image confirms the end-to-end pipeline from training through deployment-style prediction.


**What to look for in the output**

- Image filename and dimensions printed.
- Detection count with confidence threshold 0.25.
- Each detection line shows class, confidence, and four box coordinates.
- `result.show()` opens a window or inline display with drawn boxes (environment dependent).
- If no sample JPG exists, a clear message still confirms training completed.


In [ ]:
from pathlib import Path

ROOT = Path("..").resolve()
sample_dir = ROOT / "datasets" / "sample_images"
sample = next(iter(sorted(sample_dir.glob("*.jpg"))), None)

if sample:
    # predict() runs single-pass detection on one image.
    preds = model.predict(source=str(sample), imgsz=640, conf=0.25, verbose=False)
    result = preds[0]
    boxes = result.boxes

    print(f"Inference on: {sample.name}")
    print(f"Image size: {result.orig_shape[1]}x{result.orig_shape[0]} (W x H)")
    print(f"Detections (conf >= 0.25): {len(boxes)}")
    print("-" * 70)

    names = result.names
    for i, box in enumerate(boxes):
        xyxy = box.xyxy[0].tolist()
        conf = box.conf[0].item()
        cls_id = int(box.cls[0].item())
        cls_name = names[cls_id]
        print(
            f"  [{i + 1}] {cls_name:20s}  conf={conf:.3f}  "
            f"box=[{xyxy[0]:6.1f}, {xyxy[1]:6.1f}, {xyxy[2]:6.1f}, {xyxy[3]:6.1f}]"
        )

    print("-" * 70)
    result.show()
else:
    print("No sample JPG found in datasets/sample_images/.")
    print("Training and validation still completed successfully.")


### Step 5: Evaluation summary

**What this section does:** Prints a compact recap of epochs, image size, mAP values, and where weights were saved.

**Why we do it:** Absolute mAP will be modest after only 10 epochs on 128 images. The goal is to verify the Ultralytics workflow, not to match production COCO leaderboard scores.


**What to look for in the output**

- Epochs trained: 10.
- mAP values match Step 3.
- Weights saved path matches Step 2 save directory.


In [ ]:
print("YOLOv8 evaluation summary:")
print(f"  Epochs trained: {TRAIN_CONFIG['epochs']}")
print(f"  Image size:     {TRAIN_CONFIG['imgsz']}")
print(f"  mAP@0.5:        {metrics.box.map50:.4f}")
print(f"  mAP@0.5:0.95:   {metrics.box.map:.4f}")
print(f"  Weights saved:  {train_results.save_dir}")
